Drips card-transaction JSON files into `lakehouse/streaming/landing/card_txns/` once every ~1.5 seconds. Notebook **06** consumes them.

## How to run the streaming demo

1. **Start 06 first** (the consumer) so its `readStream` is watching the directory.
2. Then run **05** (this notebook) — it writes files for ~3 minutes and then exits.
3. The consumer stops on its own after 5 minutes.

## What's in each file

- 1 normal transaction (small amount, business-hours timestamp).
- Every ~25 iterations, a **fraud burst**: 3 transactions from the same card, each `> ₹20,000`, within a few seconds of each other. The consumer's velocity rule should catch these.

## Setup

In [ ]:
import sys
from pathlib import Path
if (Path.cwd() / "conf").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd() / "project" / "conf").exists():
    PROJECT_ROOT = Path.cwd() / "project"
else:
    raise RuntimeError("Run from inside project/ or repo root")
sys.path.insert(0, str(PROJECT_ROOT))

import json
import random
import shutil
import time
from datetime import datetime, timedelta
from decimal import Decimal

from conf.spark import get_spark, SILVER_DIR, STREAM_LANDING_DIR

STREAM_DIR = STREAM_LANDING_DIR / "card_txns"
shutil.rmtree(STREAM_DIR, ignore_errors=True)
STREAM_DIR.mkdir(parents=True)
print(f"streaming landing -> {STREAM_DIR}")

In [ ]:
# Load existing card IDs from the silver layer so the producer emits realistic card_id/customer_id pairs
spark = get_spark("StreamProducer")
cards = [(r["card_id"], r["customer_id"]) for r in
         spark.read.format("delta").load(str(SILVER_DIR / "card_accounts")).select("card_id", "customer_id").collect()]
print(f"loaded {len(cards)} cards")
spark.stop()  # producer doesn't need Spark beyond this point

RNG = random.Random(7)
MERCHANTS_NORMAL = [("BigBasket", "grocery"), ("Swiggy", "food"),
                    ("Amazon", "ecommerce"), ("Indian Oil", "fuel"),
                    ("Apollo Pharmacy", "health"), ("Netflix", "entertainment")]
MERCHANTS_FRAUD  = [("Unknown Merch", "other"), ("Crypto Exchange", "other"),
                    ("Foreign ATM", "cash")]

## Producer loop

In [ ]:
TOTAL_ITERATIONS = 120
SLEEP_SEC = 1.5
FRAUD_EVERY = 25  # inject a burst every N normal iterations

txn_seq = 0
files_written = 0
fraud_bursts = 0

def make_txn(card_id, customer_id, fraud=False):
    global txn_seq
    txn_seq += 1
    if fraud:
        merchant, category = RNG.choice(MERCHANTS_FRAUD)
        amount = round(RNG.uniform(20001, 95000), 2)
    else:
        merchant, category = RNG.choice(MERCHANTS_NORMAL)
        amount = round(RNG.uniform(50, 5000), 2)
    return {
        "transaction_id":    f"S{txn_seq:08d}",
        "card_id":           card_id,
        "customer_id":       customer_id,
        "merchant_name":     merchant,
        "merchant_category": category,
        "amount":            amount,
        "currency":          "INR",
        "transaction_at":    datetime.utcnow().isoformat(timespec="milliseconds"),
        "status":            "approved",
    }

def write_file(rows):
    global files_written
    fname = STREAM_DIR / f"batch_{int(time.time() * 1000)}_{files_written:05d}.json"
    # Spark JSON source expects newline-delimited JSON
    with fname.open("w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    files_written += 1

for i in range(1, TOTAL_ITERATIONS + 1):
    if i % FRAUD_EVERY == 0:
        # Compromised-card burst: 3 large txns from the same card, in one file
        cid, cust = RNG.choice(cards)
        burst = [make_txn(cid, cust, fraud=True) for _ in range(3)]
        write_file(burst)
        fraud_bursts += 1
        print(f"[{i:03d}] FRAUD BURST  card={cid:8s}  customer={cust}  files={files_written}")
    else:
        cid, cust = RNG.choice(cards)
        write_file([make_txn(cid, cust, fraud=False)])
        if i % 10 == 0:
            print(f"[{i:03d}] normal       files={files_written}  bursts so far={fraud_bursts}")
    time.sleep(SLEEP_SEC)

print(f"\nDONE. files_written={files_written}, fraud_bursts={fraud_bursts}")

## What's next

Switch back to **06-stream-fraud-detect.ipynb** — its streaming query is reading from this directory and writing alerts to `lakehouse/streaming/fraud_alerts/`.